# Week 3: Abstract Classes, Dataclasses, Typing and a Composable `DataPipeline`
## One reproducible recipe for cleaning sensor data

**Course:** AI for Aerospace & Mechanical Engineering (PhD) &nbsp;|&nbsp; **Block 0:** Python & OOP crash course

> **Provenance:** new content. The instructor's PDFs do not cover Python OOP, so the
> explanations are rewritten from (not copied from)
> [W3Schools abc Module](https://www.w3schools.com/python/ref_module_abc.asp),
> [W3Schools dataclasses Module](https://www.w3schools.com/python/ref_module_dataclasses.asp),
> [W3Schools typing Module](https://www.w3schools.com/python/ref_module_typing.asp),
> [W3Schools `__repr__()`](https://www.w3schools.com/python/python_magic_repr.asp),
> [W3Schools `__eq__()`](https://www.w3schools.com/python/python_magic_eq.asp),
> [W3Schools Iterators](https://www.w3schools.com/python/python_iterators.asp),
> [GeeksforGeeks Abstract Classes in Python](https://www.geeksforgeeks.org/python/abstract-classes-in-python/),
> [GeeksforGeeks Understanding Python Dataclasses](https://www.geeksforgeeks.org/python/understanding-python-dataclasses/),
> [GeeksforGeeks Python Generics](https://www.geeksforgeeks.org/python/python-generics/) and
> [GeeksforGeeks str() vs repr()](https://www.geeksforgeeks.org/str-vs-repr-in-python/).
> The Hampel filter is adapted from Pearson (IEEE TCST, 2002) and the MAD constant 1.4826
> from Rousseeuw & Croux (JASA, 1993).

<div dir="rtl">

**خلاصه‌ی فارسی:** دو تحلیلگر یک لاگ پرواز را با دو اسکریپت تمیز می‌کنند و دو «بیشینه‌ی ضریب بار» متفاوت گزارش می‌دهند؛ هیچ‌کس نمی‌داند کدام اسکریپت، با چه پارامترها و به چه ترتیبی اجرا شده است. در این آزمایشگاه با کلاس انتزاعی (ABC)، dataclass، نوع‌دهی (typing)، و متدهای `__repr__` و `__eq__` یک خط لوله‌ی داده (DataPipeline) می‌سازیم که مراحلش نوع‌دار، قابل مقایسه، قابل چاپ و قابل ذخیره‌اند؛ پس هر نتیجه دقیقاً به دستورالعملی که آن را ساخته قابل ردیابی است.

</div>

**How to use this notebook:** run cells top to bottom (*Restart & Run All* must work).
Cells tagged `exercise` contain `# TODO`: fill them in. A `[ ]` line means "not done yet";
`[x]` means your answer matches.

## 0. The problem: same log, same two operations, two different answers

The structures group asks for the **peak load factor** of the Week 1 test flight (the 50 Hz
UAV log). Two analysts write a cleaning script each:

* **Analyst A** smooths the accelerometer with a 5-sample moving average, then takes the peak.
* **Analyst B** first removes spikes with a 7-sample rolling median, then smooths, then takes
  the peak.

Both scripts are "reasonable". Run them.

In [1]:
from __future__ import annotations

import dataclasses
import json
import math
import sys
from abc import ABC, abstractmethod
from collections.abc import Iterator
from dataclasses import FrozenInstanceError, dataclass, field
from pathlib import Path
from typing import Any, ClassVar, Final, Generic, TypeVar

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Walk up from ``start`` until the folder that contains ``src/aerospace_ai``."""
    for folder in (start, *start.parents):
        if (folder / "src" / "aerospace_ai").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook from inside the course repository.")


REPO_ROOT: Final[Path] = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

FLIGHT_LOG: Final[Path] = REPO_ROOT / "data" / "sample" / "week01_uav_flight_log.csv"
G0: Final[float] = 9.80665                       # standard gravity [m/s^2]
ACCEL: Final[list[str]] = ["accel_x_mps2", "accel_y_mps2", "accel_z_mps2"]
TIME: Final[str] = "time_s"
SMOOTH_WINDOW: Final[int] = 5                    # samples (0.1 s at 50 Hz)
DESPIKE_WINDOW: Final[int] = 7                   # samples


def check(name: str, value: object, expected: float, tol: float) -> None:
    """Print a pass mark if ``value`` is within ``tol`` of ``expected``.

    A value of None means the exercise has not been attempted yet.
    """
    if value is None:
        print(f"[ ] {name}: not done yet")
        return
    assert abs(float(value) - expected) <= tol, f"{name}: got {value}, expected ~{expected}"
    print(f"[x] {name} = {float(value):.6g}")


def load_factor(frame: pd.DataFrame) -> np.ndarray:
    """Return n = |f| / g0 from the three accelerometer columns."""
    return np.linalg.norm(frame[ACCEL].to_numpy(dtype=float), axis=1) / G0


log = pd.read_csv(FLIGHT_LOG)
print("Repository root found:", REPO_ROOT.name, "| log shape:", log.shape)

Repository root found: AI_ML_Classrom | log shape: (14940, 10)


In [2]:
def analyst_a(frame: pd.DataFrame) -> pd.DataFrame:
    """Script A: smooth, done."""
    out = frame.copy()
    out[ACCEL] = frame[ACCEL].rolling(SMOOTH_WINDOW, center=True, min_periods=1).mean()
    return out


def analyst_b(frame: pd.DataFrame) -> pd.DataFrame:
    """Script B: despike with a rolling median, then smooth."""
    out = frame.copy()
    out[ACCEL] = frame[ACCEL].rolling(DESPIKE_WINDOW, center=True, min_periods=1).median()
    out[ACCEL] = out[ACCEL].rolling(SMOOTH_WINDOW, center=True, min_periods=1).mean()
    return out


for label, script in [("A", analyst_a), ("B", analyst_b)]:
    n = load_factor(script(log))
    k = int(np.argmax(n))
    print(f"Analyst {label}: peak n = {n[k]:.3f} g at t = {log[TIME].iloc[k]:.2f} s")

Analyst A: peak n = 1.872 g at t = 66.72 s
Analyst B: peak n = 1.801 g at t = 192.04 s


**What went wrong?** Analyst A's "peak" is a smeared sensor glitch in level cruise
(t ≈ 66.7 s): a +40 m/s² single-sample spike averaged over 5 samples still adds 8 m/s².
Analyst B finds the real pull-up at t ≈ 192 s. The engineering lesson is *despike before you
smooth*. The software lesson is harder:

1. The **order** of the steps is buried inside a function body. Nobody reading the report can
   see it.
2. The **parameters** (5, 7, `center=True`) are not recorded with the result.
3. Two recipes cannot be **compared** (`analyst_a == analyst_b` compares function objects).
4. Nothing stops a step from receiving the **wrong kind of data** until it crashes, or worse,
   until it silently returns nonsense.

This week we turn a cleaning recipe into an object: an ordered chain of **typed stages**
that prints itself (`__repr__`), compares by value (`__eq__`), checks that its stages fit
together *before* any data flows, and saves itself as plain JSON. Four tools get us there:
abstract base classes, dataclasses, type hints and generics, and the special methods.

## 1. Abstract base classes: the contract of a stage

A **stage** takes data and returns new data. Every stage must have a `transform` method; how it
transforms is the stage's own business. An **abstract base class** (ABC) writes this contract
down: a class that inherits from `ABC` and marks a method with `@abstractmethod` *cannot be
instantiated*, and neither can a subclass that forgot to implement it. The error appears at
construction time, not deep inside a run.

An ABC can also hold **concrete** methods that every stage shares (here `__call__`, which
checks the input type first) and **abstract properties** (`@property` + `@abstractmethod`),
which force every subclass to expose a value.

In [3]:
class Stage(ABC):
    """Contract: every stage transforms a DataFrame into a new DataFrame."""

    @property
    @abstractmethod
    def description(self) -> str:
        """One line that says what the stage does (abstract property)."""

    @abstractmethod
    def transform(self, data: pd.DataFrame) -> pd.DataFrame:
        """Return a new DataFrame; never modify ``data`` in place."""

    def __call__(self, data: pd.DataFrame) -> pd.DataFrame:
        """Concrete method shared by all stages: check the type, then transform."""
        if not isinstance(data, pd.DataFrame):
            raise TypeError(f"{type(self).__name__} needs a DataFrame, got {type(data).__name__}")
        return self.transform(data)


class Forgetful(Stage):
    """Implements transform but forgets the abstract property."""

    def transform(self, data: pd.DataFrame) -> pd.DataFrame:
        return data.copy()


for cls in (Stage, Forgetful):
    try:
        cls()
    except TypeError as err:
        print(f"{cls.__name__}(): TypeError: {err}")

Stage(): TypeError: Can't instantiate abstract class Stage with abstract methods description, transform
Forgetful(): TypeError: Can't instantiate abstract class Forgetful with abstract method description


**Exercise 1.1.** Physically impossible values should become NaN (missing), not be clipped:
a clipped value looks like a real measurement. Complete `PlausibleRange` so that values outside
`[lower, upper]` become NaN in the listed columns. Then use it to remove the accelerometer
spikes: an IMU configured for ±4 g cannot report more than 4 g₀ on any axis.

In [4]:
class PlausibleRange(Stage):
    """Replace values outside [lower, upper] by NaN in the given columns."""

    def __init__(self, columns: list[str], lower: float, upper: float) -> None:
        self.columns = list(columns)
        self.lower = lower
        self.upper = upper

    @property
    def description(self) -> str:
        return f"NaN outside [{self.lower:g}, {self.upper:g}] in {self.columns}"

    def transform(self, data: pd.DataFrame) -> pd.DataFrame:
        out = data.copy()
        for col in self.columns:
            out[col] = data[col].where(data[col].between(self.lower, self.upper))
        return out


plausible = PlausibleRange(ACCEL, -4 * G0, 4 * G0)
cleaned = plausible(log)
n_removed = int(cleaned[ACCEL].isna().sum().sum()) or None   # None until it works
print(plausible.description)
check("Exercise 1.1 spikes removed", n_removed, 3, 0)

NaN outside [-39.2266, 39.2266] in ['accel_x_mps2', 'accel_y_mps2', 'accel_z_mps2']
[x] Exercise 1.1 spikes removed = 3


## 2. Dataclasses: the configuration *is* the class

`PlausibleRange` above has a problem that has nothing to do with data: two objects built with
the same settings are **not equal**, and printing one shows a memory address. A stage's
fields *are* its configuration, so they should drive `__init__`, `__repr__` and `__eq__`.
`@dataclass` writes all three from the annotated class attributes.

In [5]:
class ManualMovingAverage:
    """Written by hand: no __repr__, no __eq__."""

    def __init__(self, column: str, window: int = 5) -> None:
        self.column = column
        self.window = window


@dataclass
class MovingAverage:
    """The same configuration as a dataclass: __init__, __repr__, __eq__ for free."""

    column: str
    window: int = 5


print(ManualMovingAverage("alt", 5) == ManualMovingAverage("alt", 5), ManualMovingAverage("alt"))
print(MovingAverage("alt", 5) == MovingAverage("alt", 5), MovingAverage("alt"))

False <__main__.ManualMovingAverage object at 0x7fa1b95c3650>
True MovingAverage(column='alt', window=5)


Four dataclass features matter for pipelines:

| Feature | What it does | Why a stage needs it |
|---------|--------------|----------------------|
| `field(default_factory=list)` | a fresh default per object | `columns: list = []` is refused: one list would be shared by every object (Week 2's shared-list bug) |
| `frozen=True` | assigning to a field raises `FrozenInstanceError`; objects become hashable | a recipe must not change after it produced a result |
| `__post_init__` | runs after the generated `__init__` | validate (`window` odd and ≥ 3) and normalise (list → tuple) |
| `field(init=False, repr=False, compare=False)` | a field that is not a constructor argument, not printed, not compared | **learned state**, for example a mean computed by `fit` |

A frozen dataclass cannot assign in `__post_init__` with `self.x = ...`; it uses
`object.__setattr__(self, "x", value)`, which bypasses the frozen guard exactly once, during
construction.

In [6]:
try:
    @dataclass
    class BadStage:
        columns: list = []          # mutable default: refused when the class is created
except ValueError as err:
    print("ValueError:", err)


@dataclass(frozen=True)
class Smoother:
    """Frozen, validated, normalised configuration."""

    columns: tuple[str, ...]
    window: int = 5

    def __post_init__(self) -> None:
        if self.window < 3 or self.window % 2 == 0:
            raise ValueError(f"window must be odd and >= 3, got {self.window}")
        object.__setattr__(self, "columns", tuple(self.columns))   # list in, tuple stored


s = Smoother(["accel_z_mps2"])
print(s, "| hashable:", hash(s) == hash(Smoother(("accel_z_mps2",))))
try:
    s.window = 9
except FrozenInstanceError as err:
    print("FrozenInstanceError:", err)
try:
    Smoother(["x"], window=4)
except ValueError as err:
    print("ValueError:", err)

ValueError: mutable default <class 'list'> for field columns is not allowed: use default_factory
Smoother(columns=('accel_z_mps2',), window=5) | hashable: True
FrozenInstanceError: cannot assign to field 'window'
ValueError: window must be odd and >= 3, got 4


**Exercise 2.1.** Write `Calibration` as a dataclass for a sensor's linear calibration
`y = scale * x + offset`. Requirements: frozen; `scale` must be non-zero (raise `ValueError`
in `__post_init__`); a field `fitted_rmse: float` with default `nan` that is **not** part of
`__init__`, `__repr__` or `__eq__`; and a method `apply(x)`. Two calibrations with the same scale
and offset must compare equal even if their `fitted_rmse` differ.

In [7]:
@dataclass(frozen=True)
class Calibration:
    """Linear calibration y = scale * x + offset."""

    scale: float
    offset: float = 0.0
    fitted_rmse: float = field(default=math.nan, init=False, repr=False, compare=False)

    def __post_init__(self) -> None:
        if self.scale == 0:
            raise ValueError("scale must be non-zero")

    def apply(self, x: float | np.ndarray) -> float | np.ndarray:
        """Return the calibrated value."""
        return self.scale * x + self.offset


if Calibration is None:
    print("[ ] Exercise 2.1: not done yet")
else:
    c1 = Calibration(2.0, 0.5)
    c2 = Calibration(2.0, 0.5)
    object.__setattr__(c2, "fitted_rmse", 0.03)       # simulate a different fit quality
    assert c1 == c2 and "fitted_rmse" not in repr(c1)
    check("Exercise 2.1 apply(1.0)", c1.apply(1.0), 2.5, 1e-12)

[x] Exercise 2.1 apply(1.0) = 2.5


## 3. Writing `__repr__` and `__eq__` yourself

Dataclasses cover most cases, but a container such as a pipeline needs its own. The rules:

* `__repr__` is for developers: unambiguous, ideally **valid Python** that rebuilds the object
  (`eval(repr(x)) == x`). `__str__` is for reports: short and readable. `print` uses `__str__`
  and falls back to `__repr__`.
* `__eq__(self, other)` returns `NotImplemented` (not `False`) when `other` is a different
  type; Python then tries `other.__eq__(self)`.
* If you define `__eq__`, Python sets `__hash__ = None` (unhashable) unless you also define
  `__hash__`. Equal objects **must** have equal hashes, so hash exactly the fields that `__eq__`
  compares, and only if they never change.

In [8]:
class Route:
    """An ordered list of waypoint names with hand-written special methods."""

    def __init__(self, waypoints: list[str]) -> None:
        self._waypoints = tuple(waypoints)

    def __repr__(self) -> str:
        return f"Route({list(self._waypoints)!r})"

    def __str__(self) -> str:
        return " -> ".join(self._waypoints)

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Route):
            return NotImplemented
        return self._waypoints == other._waypoints

    def __hash__(self) -> int:
        return hash(self._waypoints)            # same data as __eq__, and immutable

    def __len__(self) -> int:
        return len(self._waypoints)

    def __iter__(self) -> Iterator[str]:
        return iter(self._waypoints)


r = Route(["OIII", "OIFM", "OISS"])
print(repr(r), "|", r, "| len", len(r), "| eval round trip:", eval(repr(r)) == r)
print("compare with a list:", r == ["OIII", "OIFM", "OISS"], "| in a set:", r in {Route(["OIII", "OIFM", "OISS"])})

Route(['OIII', 'OIFM', 'OISS']) | OIII -> OIFM -> OISS | len 3 | eval round trip: True
compare with a list: False | in a set: True


**Exercise 3.1.** `Waypoint` stores latitude, longitude and altitude. Write `__repr__` (valid
Python), `__eq__` (exact comparison of all three; `NotImplemented` for other types) and a
consistent `__hash__`.

In [9]:
class Waypoint:
    """A geographic point; immutable by convention (underscore attributes)."""

    def __init__(self, lat_deg: float, lon_deg: float, alt_m: float = 0.0) -> None:
        self._lat, self._lon, self._alt = float(lat_deg), float(lon_deg), float(alt_m)

    def __repr__(self) -> str:
        return f"Waypoint({self._lat!r}, {self._lon!r}, {self._alt!r})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Waypoint):
            return NotImplemented
        return (self._lat, self._lon, self._alt) == (other._lat, other._lon, other._alt)

    def __hash__(self) -> int:
        return hash((self._lat, self._lon, self._alt))


w = Waypoint(35.6892, 51.3134, 1190.0)
try:
    ok = (eval(repr(w)) == w) and (w == Waypoint(35.6892, 51.3134, 1190.0)) and len({w, Waypoint(35.6892, 51.3134, 1190.0)}) == 1
except Exception:          # eval of the default "<...object at 0x...>" repr fails
    ok = False
check("Exercise 3.1 special methods", 1 if ok else None, 1, 0)

[x] Exercise 3.1 special methods = 1


## 4. Type hints, generics and why the pipeline still checks at run time

Type hints document intent and let tools such as `mypy` find mistakes before running, but
**Python does not enforce them**: the function below accepts a string happily.

In [10]:
def peak_g(accel_z: np.ndarray) -> float:
    """Return the peak of accel_z in units of g0."""
    return float(np.max(accel_z) / G0)


try:
    print(peak_g("9.8"))                 # the hint says ndarray; Python does not care
except TypeError as err:
    print("fails only when the body runs:", err)

fails only when the body runs: ufunc 'maximum' did not contain a loop with signature matching types (dtype('<U3'), dtype('<U3')) -> None


A stage maps an input type to an output type. **Generics** let us say that once, with type
variables: `Stage[InT, OutT]`. A `TypeVar` is a placeholder; `Generic[InT, OutT]` makes the class
accept two type parameters, exactly like `list[int]` or `dict[str, float]`.

At run time these parameters are **erased**: `Stage[pd.DataFrame, np.ndarray]` and
`Stage[int, str]` are the same class object. So the course library also stores the types as
plain class attributes, `INPUT_TYPE` and `OUTPUT_TYPE` (`ClassVar`: shared by the class, not a
dataclass field), and checks a chain with `issubclass` when the pipeline is **built**.

In [11]:
InT = TypeVar("InT")
OutT = TypeVar("OutT")


class TypedStage(ABC, Generic[InT, OutT]):
    """Minimal generic stage: the type parameters are for readers and mypy."""

    INPUT_TYPE: ClassVar[type] = object
    OUTPUT_TYPE: ClassVar[type] = object

    @abstractmethod
    def transform(self, data: InT) -> OutT:
        """Map InT to OutT."""


class FrameToArray(TypedStage[pd.DataFrame, np.ndarray]):
    INPUT_TYPE = pd.DataFrame
    OUTPUT_TYPE = np.ndarray

    def transform(self, data: pd.DataFrame) -> np.ndarray:
        return data.to_numpy(dtype=float)


print(FrameToArray.__orig_bases__)                 # what we wrote
print(FrameToArray.INPUT_TYPE.__name__, "->", FrameToArray.OUTPUT_TYPE.__name__)   # what survives at run time

(__main__.TypedStage[pandas.core.frame.DataFrame, numpy.ndarray],)
DataFrame -> ndarray


**Exercise 4.1.** Write `can_follow(left, right)`: True if stage class `right` can accept
what stage class `left` produces. Use `issubclass` on the declared types. Note the direction: a
stage that outputs `bool` can feed a stage that accepts `int`, because `bool` is a subclass of
`int`, but not the other way round.

In [12]:
def can_follow(left: type[TypedStage[Any, Any]], right: type[TypedStage[Any, Any]]) -> bool | None:
    """Return True if ``right`` accepts the output type of ``left``."""
    return issubclass(left.OUTPUT_TYPE, right.INPUT_TYPE)


class ArrayToFrame(TypedStage[np.ndarray, pd.DataFrame]):
    INPUT_TYPE = np.ndarray
    OUTPUT_TYPE = pd.DataFrame

    def transform(self, data: np.ndarray) -> pd.DataFrame:
        return pd.DataFrame(data)


answers = (can_follow(FrameToArray, ArrayToFrame), can_follow(FrameToArray, FrameToArray))
check("Exercise 4.1", None if None in answers else int(answers == (True, False)), 1, 0)

[x] Exercise 4.1 = 1


## 5. Worked Example 1 (by hand first): the Hampel filter

A rolling **median** ignores a spike but also flattens real peaks. The **Hampel filter** only
replaces a sample when it is far from its neighbours, measured robustly. For sample $x_k$ and a
centred window of $w$ samples:

$$m_k = \operatorname{median}(\text{window}),\qquad
\text{MAD}_k = \operatorname{median}\lvert x_j - m_k\rvert,\qquad
x_k \text{ is an outlier if } \lvert x_k - m_k\rvert > n_\sigma \cdot 1.4826\cdot\text{MAD}_k .$$

The factor 1.4826 turns the MAD into an estimate of the standard deviation for Gaussian noise.

**By hand.** A barometric altitude record (m), one spike at $k=3$, window $w=5$, $n_\sigma=3$:

| $k$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| $h_k$ | 100.0 | 100.4 | 99.8 | **106.0** | 100.3 | 100.1 | 99.9 |

* Window at $k=3$: {100.4, 99.8, 106.0, 100.3, 100.1}; sorted 99.8, 100.1, **100.3**, 100.4, 106.0, so $m_3 = 100.3$.
* Deviations: 0.1, 0.5, 5.7, 0.0, 0.2; sorted 0.0, 0.1, **0.2**, 0.5, 5.7, so $\text{MAD}_3 = 0.2$.
* Threshold $3 \times 1.4826 \times 0.2 = 0.8896$ m; $\lvert 106.0 - 100.3\rvert = 5.7 > 0.8896$: replace by 100.3.
* At $k=4$: window {99.8, 106.0, 100.3, 100.1, 99.9}, $m_4 = 100.1$, deviations 0.3, 5.9, 0.2, 0.0, 0.2, MAD = 0.2;
  $\lvert 100.3 - 100.1\rvert = 0.2 < 0.8896$: keep. The spike inflates nothing because medians ignore it.

In [13]:
from aerospace_ai.pipeline import HampelFilter

altitude = pd.DataFrame({"timestamp": np.arange(7.0), "alt_m": [100.0, 100.4, 99.8, 106.0, 100.3, 100.1, 99.9]})
hampel = HampelFilter(["alt_m"], window=5, n_sigmas=3.0)
is_outlier, local_median = hampel.outlier_mask(altitude["alt_m"])
print(hampel)
print("outliers at k =", np.flatnonzero(is_outlier), "| m_3 =", local_median[3])
print("filtered:", hampel(altitude)["alt_m"].tolist())
assert np.flatnonzero(is_outlier).tolist() == [3] and math.isclose(local_median[3], 100.3)

HampelFilter(columns=('alt_m',), window=5, n_sigmas=3.0)
outliers at k = [3] | m_3 = 100.3
filtered: [100.0, 100.4, 99.8, 100.3, 100.3, 100.1, 99.9]


## 6. Worked Example 2 (by hand first): the order of stages matters

Take the step $x = [1, 1, 1, 10, 1, 1, 1]$ (a one-sample glitch of +9), a Hampel filter with
$w=3$ and a centred 3-sample mean (truncated at the ends).

**Despike, then smooth.** At $k=3$ the window {1, 10, 1} has median 1 and deviations
{0, 9, 0}, so MAD = 0 and the threshold is 0: $\lvert 10-1\rvert > 0$, replaced by 1.
Everywhere else the sample equals its median and is kept. The mean of a constant is the constant:
**peak = 1**.

**Smooth, then despike.** The mean spreads the glitch: $k=2,3,4$ become $(1+1+10)/3 = 4$, so
the signal is $[1, 1, 4, 4, 4, 1, 1]$. Now every window around the bump, e.g. {1, 4, 4} at $k=2$,
has median 4: nothing is an outlier. **Peak = 4**.

Composition of stages is not commutative: $H\circ S \ne S\circ H$. A pipeline must therefore
record its order, and two pipelines with the same stages in a different order must compare
**unequal**.

In [14]:
from aerospace_ai.pipeline import RollingMean

step = pd.DataFrame({"x": [1.0, 1.0, 1.0, 10.0, 1.0, 1.0, 1.0]})
despike_then_smooth = HampelFilter(["x"], window=3) >> RollingMean(["x"], window=3)
smooth_then_despike = RollingMean(["x"], window=3) >> HampelFilter(["x"], window=3)
print("H then S:", despike_then_smooth(step)["x"].round(3).tolist())
print("S then H:", smooth_then_despike(step)["x"].round(3).tolist())
print("same recipe?", despike_then_smooth == smooth_then_despike)

H then S: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


S then H: [1.0, 1.0, 4.0, 4.0, 4.0, 1.0, 1.0]
same recipe? False


## 7. The course library: `aerospace_ai.pipeline`

`src/aerospace_ai/pipeline.py` puts sections 1 to 6 together:

* `PipelineStage(ABC, Generic[InT, OutT])` with `INPUT_TYPE`/`OUTPUT_TYPE`, abstract `transform`,
  optional `fit`, a concrete `__call__` that checks types, and a registry filled by
  `__init_subclass__` (every stage class registers itself by name when it is defined).
* Stages as dataclasses: `SensorToFrame`, `SelectColumns`, `DropNonFinite`, `FillGaps`,
  `HampelFilter`, `RollingMean`, `TimeDerivative`, `Standardize` (learns mean and std), `ToArray`.
* `DataPipeline`: built with `>>`; immutable; checks neighbours in `__init__`; hand-written
  `__repr__`, `__str__`, `__eq__`, `__len__`, `__iter__`, `__getitem__`; `run` returns the
  output plus one `StageReport` per stage; `to_config`/`from_config` round-trip through JSON.

In [15]:
from aerospace_ai.pipeline import (
    DataPipeline, DropNonFinite, FillGaps, NotFittedError, PipelineStage, SelectColumns,
    SensorToFrame, Standardize, TimeDerivative, ToArray,
)

recipe = (
    SelectColumns([TIME, *ACCEL])
    >> HampelFilter(ACCEL, window=DESPIKE_WINDOW)
    >> RollingMean(ACCEL, window=SMOOTH_WINDOW)
)
print(repr(recipe))
print("str:", recipe, "| stages:", len(recipe), "| second stage:", recipe[1].name)
print("input", recipe.input_type.__name__, "-> output", recipe.output_type.__name__)
print("registered stage classes:", sorted(PipelineStage.registry)[:6], "...")

DataPipeline(
    SelectColumns(columns=('time_s', 'accel_x_mps2', 'accel_y_mps2', 'accel_z_mps2'))
    >> HampelFilter(columns=('accel_x_mps2', 'accel_y_mps2', 'accel_z_mps2'), window=7, n_sigmas=3.0)
    >> RollingMean(columns=('accel_x_mps2', 'accel_y_mps2', 'accel_z_mps2'), window=5)
)
str: SelectColumns -> HampelFilter -> RollingMean | stages: 3 | second stage: HampelFilter
input DataFrame -> output DataFrame
registered stage classes: ['DropNonFinite', 'FillGaps', 'HampelFilter', 'RollingMean', 'SelectColumns', 'SensorToFrame'] ...


In [16]:
# Type errors are caught when the pipeline is BUILT, before any data flows.
for build in (
    lambda: ToArray(ACCEL) >> RollingMean(ACCEL),
    lambda: RollingMean(ACCEL) >> SensorToFrame(),
    lambda: DataPipeline([RollingMean(ACCEL), "not a stage"]),
):
    try:
        build()
    except TypeError as err:
        print("TypeError:", err)

TypeError: step 0 ToArray outputs ndarray, but step 1 RollingMean expects DataFrame
TypeError: step 0 RollingMean outputs DataFrame, but step 1 SensorToFrame expects BaseSensor
TypeError: item 1 is str, not a PipelineStage


## 8. Back to the flight log: both analysts as pipelines

Written as pipelines, the two scripts become two values that print, compare and save.
(Pipeline B uses the Hampel filter of Section 5 instead of a plain rolling median: it only
replaces samples that are far from their neighbours, so the peak moves from 1.801 to 1.803 g.)
`run` also returns an audit trail: rows in and out, NaN count, how many cells each stage
changed and how long it took.

In [17]:
pipeline_a = RollingMean(ACCEL, window=SMOOTH_WINDOW) >> SelectColumns([TIME, *ACCEL])
pipeline_b = (
    HampelFilter(ACCEL, window=DESPIKE_WINDOW)
    >> RollingMean(ACCEL, window=SMOOTH_WINDOW)
    >> SelectColumns([TIME, *ACCEL])
)
result_a, result_b = pipeline_a.run(log), pipeline_b.run(log)
peaks = {}
for label, result in [("A", result_a), ("B", result_b)]:
    n = load_factor(result.output)
    k = int(np.argmax(n))
    peaks[label] = (n[k], result.output[TIME].iloc[k])
    print(f"Pipeline {label}: {result.config['stages'][0]['stage']:<12} first | peak n = {n[k]:.3f} g at t = {peaks[label][1]:.2f} s")
print("A == B?", pipeline_a == pipeline_b)
result_b.report_frame().round(4)

Pipeline A: RollingMean  first | peak n = 1.872 g at t = 66.72 s
Pipeline B: HampelFilter first | peak n = 1.803 g at t = 192.00 s
A == B? False


,stage,rows_in,rows_out,nan_out,cells_changed,seconds
step,,,,,,
0,HampelFilter,14940,14940,150,2140,0.0854
1,RollingMean,14940,14940,150,44815,0.0228
2,SelectColumns,14940,14940,0,0,0.0056


In [18]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(log[TIME], load_factor(log), lw=0.4, color="0.7", label="raw (3 spikes)")
ax.plot(log[TIME], load_factor(result_a.output), lw=0.7, label=f"A: {pipeline_a}")
ax.plot(log[TIME], load_factor(result_b.output), lw=0.7, label=f"B: {pipeline_b}")
for label, (peak, t_peak) in peaks.items():
    ax.annotate(f"{label}: {peak:.2f} g", (t_peak, peak), textcoords="offset points", xytext=(5, 8))
ax.set(xlabel="time [s]", ylabel="load factor n [g]", ylim=(0.6, 2.2), title="Same log, two recipes")
ax.legend(loc="upper left", fontsize=8)
plt.show()

<Figure size 1000x350 with 1 Axes>

**Reproducibility.** Save the configuration next to every number you report. Anyone can
rebuild the exact recipe from the JSON text, and `==` proves it is the same one.

In [19]:
config_text = json.dumps(result_b.config, indent=1)
print(config_text[:400], "...")
rebuilt = DataPipeline.from_config(json.loads(config_text))
print("rebuilt == pipeline_b:", rebuilt == pipeline_b)
pd.testing.assert_frame_equal(rebuilt(log), result_b.output)

{
 "version": 1,
 "stages": [
  {
   "stage": "HampelFilter",
   "params": {
    "columns": [
     "accel_x_mps2",
     "accel_y_mps2",
     "accel_z_mps2"
    ],
    "window": 7,
    "n_sigmas": 3.0
   }
  },
  {
   "stage": "RollingMean",
   "params": {
    "columns": [
     "accel_x_mps2",
     "accel_y_mps2",
     "accel_z_mps2"
    ],
    "window": 5
   }
  },
  {
   "stage": "SelectColumns", ...
rebuilt == pipeline_b: True


**Exercise 8.1.** Build a **climb-rate** pipeline for the barometric altitude `baro_alt_m`:
fill NaN runs of up to 5 samples (`FillGaps`, time column `time_s`), despike (`HampelFilter`,
window 7), smooth over about 1 s (`RollingMean`, window 51) and differentiate
(`TimeDerivative` into a new column `climb_mps`). Report the maximum climb rate and the time at
which it occurs. Then look at Section 0 of the Week 1 notebook: the log has a 1.2 s logger gap
at t ≈ 100 s. Does your maximum look physical? (Practice Assignment 3 fixes this with a
resampling stage.)

In [20]:
climb_pipeline = (
    FillGaps(["baro_alt_m"], max_gap=5, time_column=TIME)
    >> HampelFilter(["baro_alt_m"], window=7)
    >> RollingMean(["baro_alt_m"], window=51)
    >> TimeDerivative("baro_alt_m", "climb_mps", time_column=TIME)
)
max_climb = t_max_climb = None

if climb_pipeline is not None:
    climb = climb_pipeline(log)
    k = int(climb["climb_mps"].to_numpy().argmax())
    max_climb, t_max_climb = climb["climb_mps"].iloc[k], climb[TIME].iloc[k]
    print(climb_pipeline)
    print(f"max climb {max_climb:.2f} m/s at t = {t_max_climb:.2f} s")
check("Exercise 8.1 max climb rate [m/s]", max_climb, 20.985, 0.01)

FillGaps -> HampelFilter -> RollingMean -> TimeDerivative
max climb 20.99 m/s at t = 99.76 s
[x] Exercise 8.1 max climb rate [m/s] = 20.9853


*Answer to the question:* the maximum sits at t ≈ 99.8 s, right before the gap. `RollingMean`
counts **rows**, not seconds, so the 51-row window around t = 99.8 s reaches 1.2 s further in
time on one side than on the other, and the altitude jump across the gap leaks into the slope.
A uniform time grid (resampling) first is the fix.

## 9. Worked Example 3 (by hand first): a stage that learns, and leakage

`Standardize` computes $z = (x-\mu)/\sigma$. The mean and standard deviation are **learned**
by `fit` and reused by `transform`. Learn them from the training data only.

**By hand.** Training altitudes $\{2, 4, 6\}$ m: $\mu = 4$, $\sigma = \sqrt{(4+0+4)/3} = 1.633$ m.
A new test sample $x = 8$ m: $z = (8-4)/1.633 = 2.449$, clearly outside the training range,
which is information the model needs. If we had fitted on train **and** test,
$\{2,4,6,8\}$: $\mu = 5$, $\sigma = \sqrt{(9+1+1+9)/4} = 2.236$, and $z = 1.342$: the test sample
now looks ordinary because it helped define "ordinary". That is **data leakage** (Week 5).

In the class, `mean_` and `std_` are declared with `field(init=False, repr=False, compare=False)`:
learned state is not configuration, so a fitted and an unfitted `Standardize(["x"])` are the
same recipe.

In [21]:
train, test = pd.DataFrame({"x": [2.0, 4.0, 6.0]}), pd.DataFrame({"x": [8.0]})
scaler = Standardize(["x"])
try:
    scaler(test)
except NotFittedError as err:
    print("NotFittedError:", err)
scaler.fit(train)
z_right = scaler(test)["x"].iloc[0]
z_leaky = Standardize(["x"]).fit(pd.concat([train, test]))(test)["x"].iloc[0]
print(f"fit on train: z = {z_right:.3f} | fit on train+test: z = {z_leaky:.3f}")
print(scaler, "| fitted:", scaler.is_fitted, "| equal to a fresh one:", scaler == Standardize(["x"]))
assert math.isclose(z_right, 2.449, abs_tol=1e-3) and math.isclose(z_leaky, 1.342, abs_tol=1e-3)

NotFittedError: Standardize must be fitted before transform
fit on train: z = 2.449 | fit on train+test: z = 1.342
Standardize(columns=('x',)) | fitted: True | equal to a fresh one: True


## 10. Any sensor in, a model-ready array out

`SensorToFrame` accepts **any** `BaseSensor`: it only uses the shared interface from Week 2
(`to_frame`). That is why the pipeline does not need to know about LiDAR or GPS: your
Assignment 1 sensors plug in the same way as `IMUSensor` and `BarometerSensor`.

**Exercise 10.1.** Load the flight log into an `IMUSensor` configured for ±4 g (Week 2,
Section 9: the 3 spikes are rejected as saturated). Build a pipeline
`SensorToFrame >> HampelFilter >> RollingMean >> ToArray` for the three accelerometer channels
(windows 7 and 5) and compute the peak load factor from the output array with
`np.linalg.norm(..., axis=1) / G0`.

In [22]:
from aerospace_ai.sensors import IMUSensor

imu = IMUSensor("imu_flight", rate_hz=50.0, accel_range_mps2=4 * G0)
imu.add_readings(log[TIME].to_numpy(), log[list(IMUSensor.CHANNELS)].to_numpy())
print(imu)

imu_pipeline = (
    SensorToFrame()
    >> HampelFilter(ACCEL, window=DESPIKE_WINDOW)
    >> RollingMean(ACCEL, window=SMOOTH_WINDOW)
    >> ToArray(ACCEL)
)
peak_from_sensor = None
if imu_pipeline is not None:
    features = imu_pipeline(imu)
    peak_from_sensor = (np.linalg.norm(features, axis=1) / G0).max()
    print("features:", features.shape, "| peak n:", round(peak_from_sensor, 3))
check("Exercise 10.1 peak load factor [g]", peak_from_sensor, 1.803, 0.002)

IMUSensor(sensor_id='imu_flight', rate_hz=50.0, readings=14940, rejected=3)
features: (14937, 3) | peak n: 1.803
[x] Exercise 10.1 peak load factor [g] = 1.80292


## 11. Final checks

This is the notebook's own test cell: it verifies the worked examples and the flight-log
result and must pass on every machine. It also runs the library's pytest suite.

In [23]:
import subprocess

assert peaks["A"][0] > 1.85 and 66.0 < peaks["A"][1] < 67.5        # A: a smeared glitch
assert 1.79 < peaks["B"][0] < 1.82 and 190.0 < peaks["B"][1] < 194.0  # B: the real pull-up
assert pipeline_a != pipeline_b and rebuilt == pipeline_b
assert despike_then_smooth(step)["x"].max() == 1.0 and math.isclose(smooth_then_despike(step)["x"].max(), 4.0)
assert hampel(altitude)["alt_m"].iloc[3] == 100.3
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "-p", "no:warnings",
     "tests/test_pipeline.py"],
    cwd=REPO_ROOT, capture_output=True, text=True,
)
print(result.stdout.strip().splitlines()[-1])
assert result.returncode == 0, result.stdout
print("All Week 3 checks passed.")

36 passed in 1.65s
All Week 3 checks passed.


## What we now know how to do

* Write a contract as an abstract base class, with abstract methods, abstract properties and
  shared concrete methods; know that a missing method fails at construction.
* Describe configuration with `@dataclass`: defaults, `default_factory`, `frozen=True`,
  `__post_init__` validation, and `field(init=False, repr=False, compare=False)` for learned state.
* Write `__repr__`, `__str__`, `__eq__` (with `NotImplemented`) and a consistent `__hash__`.
* Use type hints, `TypeVar` and `Generic`, and know they are not enforced at run time.
* Despike with a Hampel filter by hand and in code; prove that stage order matters.
* Compose typed stages into a `DataPipeline` that type-checks itself, reports what each stage
  did, and round-trips through JSON for reproducibility.
* Fit learned statistics on training data only.

**Practice Assignment 3** (`assignments/practice03/`): write three new stages (one of them
learns), a resampling stage that fixes the logger-gap artefact, and a saved, reproducible
air-data pipeline.
**Next week:** design patterns for ML (Strategy, Observer, Factory) and pytest in depth; the
stage registry you met here becomes a Factory.